import libraries like unsloth

In [1]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
# Install Unsloth and dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

model selection llama 3

In [8]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset, concatenate_datasets # Import necessary functions
import json

# 1. Configuration
max_seq_length = 2048 # Supports RoPE Scaling internally
dtype = None # Auto-detects GPU capabilities (Float16 for T4, Bfloat16 for Ampere)
load_in_4bit = True # 4bit quantization to reduce memory usage by 4x

# 2. Load the Pre-trained Model (Llama-3 8B)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. Load and Combine Your Specific Datasets
file_names = ["constitution_qa.json", "ipc_qa.json", "crpc_qa.json"]

print("Loading and merging datasets...")

loaded_datasets = []
for file in file_names:
    try:
        # Load each JSON file using datasets library
        current_dataset = load_dataset("json", data_files=file, split="train")
        source_tag = file.replace("_qa.json", "").upper() # e.g., "IPC", "CONSTITUTION"

        # Add a 'source' column to each dataset
        current_dataset = current_dataset.map(lambda example: {'source': source_tag})
        loaded_datasets.append(current_dataset)
        print(f"Loaded {len(current_dataset)} records from {file}")
    except Exception as e:
        print(f"Warning: Could not load {file}. Error: {e}")

# Concatenate all loaded datasets
if loaded_datasets:
    dataset = concatenate_datasets(loaded_datasets)
    print(f"Total training records: {len(dataset)}")
else:
    dataset = None # Ensure dataset is None if no data was loaded
    print("No datasets loaded. Total training records: 0")


==((====))==  Unsloth 2026.1.2: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading and merging datasets...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/4082 [00:00<?, ? examples/s]

Loaded 4082 records from constitution_qa.json


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2267 [00:00<?, ? examples/s]

Loaded 2267 records from ipc_qa.json


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/8194 [00:00<?, ? examples/s]

Loaded 8194 records from crpc_qa.json
Total training records: 14543


data preparation

In [9]:
# Llama-3 Chat Format
legal_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert Indian Legal Assistant. You answer questions based on the Constitution of India, IPC (Indian Penal Code), and CrPC (Code of Criminal Procedure).
Answer strictly based on the provided context.<|eot_id|><|start_header_id|>user<|end_header_id|>

Source Law: {}
Question: {}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    sources   = examples["source"]
    questions = examples["question"]
    answers   = examples["answer"]
    texts = []
    for source, question, answer in zip(sources, questions, answers):
        # Format the text
        text = legal_prompt.format(source, question, answer) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

# Map the formatting function to the dataset
dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/14543 [00:00<?, ? examples/s]

configure lora

In [11]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Higher rank for complex reasoning (Standard is 16)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

training

In [12]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2, # Small batch size to save RAM
        gradient_accumulation_steps = 4, # Accumulate gradients to simulate larger batch
        warmup_steps = 5,
        max_steps = 100, # Set to 0 or None for full training (approx 1 epoch)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # 8-bit optimizer to save memory
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "legal_ai_outputs",
    ),
)

print("Starting Training...")
trainer_stats = trainer.train()
print("Training Complete!")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Starting Training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14,543 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


...

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.867400
2,4.183700
3,3.539500
4,3.465500
5,2.501300
6,2.186400
7,1.753700
8,1.656000
9,1.649400
10,1.685600


wandb: WARNING URL not available in offline run


train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇█
train/global_step,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/grad_norm,▄▃▃█▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▂▇████▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁
train/loss,▇█▇▄▄▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_flos,4522870497705984.0
train/epoch,0.05501
train/global_step,100
train/grad_norm,0.92587
train/learning_rate,0.0
train/loss,1.0962


Training Complete!


inference

In [ ]:
FastLanguageModel.for_inference(model) # Enable fast inference

# Define a test question
test_source = "IPC"
test_question = "bad comments on women punishments?"

# Prepare the input
input_text = legal_prompt.format(test_source, test_question, "") # Leave answer blank

inputs = tokenizer([input_text], return_tensors = "pt").to("cuda")

# Generate response
outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
response = tokenizer.batch_decode(outputs)[0]

# Extract and print only the assistant's answer
print("Legal AI Answer:")
print(response.split("<|start_header_id|>assistant<|end_header_id|>")[-1].replace(EOS_TOKEN, ""))

Legal AI Answer:


The punishment for a person who makes or publishes any imputation on the character of a deceased woman is imprisonment for a term which may extend to two years, or with fine, or with both.


saving the model

In [14]:
model.save_pretrained("Legal_AI_Model_LoRA")
tokenizer.save_pretrained("Legal_AI_Model_LoRA")
print("Model saved to local folder 'Legal_AI_Model_LoRA'")

Model saved to local folder 'Legal_AI_Model_LoRA'
